# 02 — Preprocessing

目標：標準化特徵、切分訓練測試集、處理 class imbalance，輸出乾淨的資料到 `data/processed/`。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib
import os

## 1. 載入資料

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')
print(df.shape)

## 2. 標準化 Amount 與 Time

In [ ]:
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])
df = df.drop(columns=['Amount', 'Time'])
df.head()

## 3. Train / Test Split

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')
print(f'Train fraud rate: {y_train.mean():.4f}')
print(f'Test fraud rate:  {y_test.mean():.4f}')

## 4. 處理 Class Imbalance（SMOTE）

> SMOTE 只套用在訓練集，測試集保持原始分布以反映真實情境。

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Resampled train size: {X_train_res.shape}')
print(f'Resampled fraud rate: {y_train_res.mean():.4f}')

## 5. 儲存處理後的資料

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

pd.DataFrame(X_train_res, columns=X.columns).assign(Class=y_train_res.values).to_csv('../data/processed/train.csv', index=False)
pd.DataFrame(X_test, columns=X.columns).assign(Class=y_test.values).to_csv('../data/processed/test.csv', index=False)

print('Saved: data/processed/train.csv')
print('Saved: data/processed/test.csv')